# Optuna — task-indexed cheatsheet


Optuna is hyperparameter search done well. Three things make it nicer than `GridSearchCV`:

1. **Smart sampling** (TPE) — focuses budget on promising regions instead of brute-forcing the grid.
2. **Pruning** — aborts trials that are clearly losing early, freeing budget for more samples.
3. **Persistence** — every trial is recorded; you can resume across runs / processes via SQLite.

The price: you write a small `objective(trial)` function instead of declaring a parameter grid. The flexibility is worth it.


---
## Setup

Run this once.


### Setup — run me first


In [1]:
import numpy as np
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from sklearn.datasets import make_regression
import lightgbm as lgb

optuna.logging.set_verbosity(optuna.logging.WARNING)
rng = np.random.default_rng(0)
X, y = make_regression(n_samples=400, n_features=4, noise=5, random_state=0)

/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 1. The basic study

An Optuna study is just a `create_study` + `optimize(objective, n_trials)` pair. Inside the objective, you ask the trial for parameter values and return a score.


### How do I write a minimal Optuna study?

`create_study(direction='maximize')`, define an objective taking a `trial`, call `study.optimize`.


In [2]:
# Toy objective: maximise -(x-3)^2 — best x = 3.
def obj(trial):
    x = trial.suggest_float('x', -10, 10)
    return -(x - 3) ** 2

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(obj, n_trials=20, show_progress_bar=False)
print(f'best x: {study.best_params["x"]:.3f}    best value: {study.best_value:.4f}')

best x: 2.918    best value: -0.0067


*When to use 'maximize' vs 'minimize'*: depends on the metric — accuracy maximize, MAE minimize. *Common mistake*: forgetting to set `seed=` on the sampler — runs aren't reproducible without it.


### How do I declare a search space?

`trial.suggest_int / suggest_float / suggest_categorical / suggest_loguniform` (legacy alias).


In [3]:
def obj_full(trial):
    return trial.suggest_int('a', 1, 10) + \
           trial.suggest_float('b', 0.0, 1.0) + \
           trial.suggest_float('c', 1e-3, 1e+0, log=True) * 100 + \
           {'small': 0, 'medium': 1, 'large': 2}[trial.suggest_categorical('size', ['small', 'medium', 'large'])]

study = optuna.create_study(direction='maximize')
study.optimize(obj_full, n_trials=10, show_progress_bar=False)
print(f'best params: {study.best_params}')

best params: {'a': 3, 'b': 0.2649620320563101, 'c': 0.7309473160262723, 'size': 'medium'}


*When to use log=True*: any time the parameter spans orders of magnitude (learning rates, regularisation strengths). *Common mistake*: using a uniform float for learning rate from 0 to 0.1 — almost all the probability mass is wasted on values too high to be useful.


---
## 2. A realistic ML objective

The pattern: cross-validate inside the objective, return mean fold score. Use `TimeSeriesSplit` for sequential data.


### How do I tune a LightGBM model?

Suggest hyperparameters, fit through TimeSeriesSplit, return mean CV score.


In [4]:
def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 400),
        learning_rate=trial.suggest_float('lr', 0.01, 0.2, log=True),
        num_leaves=trial.suggest_int('num_leaves', 15, 127),
        min_child_samples=trial.suggest_int('min_child_samples', 5, 100),
        verbosity=-1, random_state=0,
    )
    scores = []
    for tr, va in TimeSeriesSplit(3).split(X):
        m = lgb.LGBMRegressor(**params).fit(X[tr], y[tr])
        scores.append(mean_absolute_error(y[va], m.predict(X[va])))
    return float(np.mean(scores))

study = optuna.create_study(direction='minimize',
                             sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=10, show_progress_bar=False)
print(f'best CV MAE: {study.best_value:.4f}')
print(f'best params: {study.best_params}')

/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2

best CV MAE: 19.8977
best params: {'n_estimators': 270, 'lr': 0.16004016494233675, 'num_leaves': 23, 'min_child_samples': 13}


/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2

*Why TimeSeriesSplit inside the objective*: the search needs to evaluate generalisation, not just training fit. *Common mistake*: scoring against a single held-out val set — the search overfits the val set noise across many trials.


---
## 3. Pruning — abort losing trials early

Report intermediate scores via `trial.report(value, step=k)` and check `trial.should_prune()`. Combined with `MedianPruner`, this halves your wall time.


### How do I add pruning to a CV objective?

Report mean-of-folds-so-far at each fold; raise `optuna.TrialPruned` if the pruner says so.


In [5]:
def obj_pruned(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 400),
        learning_rate=trial.suggest_float('lr', 0.01, 0.2, log=True),
        num_leaves=trial.suggest_int('num_leaves', 15, 127),
        verbosity=-1, random_state=0,
    )
    scores = []
    for k, (tr, va) in enumerate(TimeSeriesSplit(5).split(X)):
        m = lgb.LGBMRegressor(**params).fit(X[tr], y[tr])
        scores.append(mean_absolute_error(y[va], m.predict(X[va])))
        trial.report(np.mean(scores), step=k)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(scores))

study = optuna.create_study(direction='minimize',
                             pruner=optuna.pruners.MedianPruner(n_startup_trials=3),
                             sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(obj_pruned, n_trials=15, show_progress_bar=False)
n_pruned = sum(1 for t in study.trials if t.state.name == 'PRUNED')
print(f'completed: {len(study.trials) - n_pruned}/{len(study.trials)}, pruned: {n_pruned}')
print(f'best: {study.best_value:.4f}')

/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2

/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2

/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2

/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2

/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2

completed: 5/15, pruned: 10
best: 21.9303


/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


*Why n_startup_trials=3*: the median pruner needs a baseline; pruning the very first trials makes no sense. *Common mistake*: setting n_startup_trials=0 — first promising trial gets pruned because there's nothing to compare against.


---
## 4. Persistence — resume across processes

SQLite-backed studies survive crashes and can be opened by multiple processes for parallel search.


### How do I save and resume a study?

Pass `storage='sqlite:///path.db'` and a stable `study_name=`. Reload with `optuna.load_study`.


In [6]:
import os, tempfile

db = os.path.join(tempfile.gettempdir(), 'optuna_demo.db')
if os.path.exists(db): os.remove(db)
storage = f'sqlite:///{db}'

s1 = optuna.create_study(direction='minimize', storage=storage,
                         study_name='demo', sampler=optuna.samplers.TPESampler(seed=0))
s1.optimize(objective, n_trials=3, show_progress_bar=False)
print(f'after run 1: {len(s1.trials)} trials, best {s1.best_value:.4f}')

# Reload from disk and continue.
s2 = optuna.load_study(study_name='demo', storage=storage)
s2.optimize(objective, n_trials=3, show_progress_bar=False)
print(f'after run 2: {len(s2.trials)} trials, best {s2.best_value:.4f}')

/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2

/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2

after run 1: 3 trials, best 47.8796


after run 2: 6 trials, best 32.1606


/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


*When to use*: long searches that need to survive a kernel restart, distributed search across machines. *Common mistake*: forgetting `study_name=` and ending up with the default — confusing if you have multiple studies in the same DB.


---
## 5. Multi-objective optimisation

When you care about more than one metric (accuracy AND fit time, for example), return a tuple and use `directions=`.


### How do I optimise two objectives at once?

`create_study(directions=['minimize', 'minimize'])`. Objective returns a tuple. Result is a Pareto front.


In [7]:
import time

def obj_multi(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        learning_rate=trial.suggest_float('lr', 0.01, 0.2, log=True),
        num_leaves=trial.suggest_int('num_leaves', 15, 127),
        verbosity=-1, random_state=0,
    )
    t0 = time.time()
    scores = []
    for tr, va in TimeSeriesSplit(2).split(X):
        m = lgb.LGBMRegressor(**params).fit(X[tr], y[tr])
        scores.append(mean_absolute_error(y[va], m.predict(X[va])))
    fit_seconds = time.time() - t0
    return float(np.mean(scores)), float(fit_seconds)

study = optuna.create_study(directions=['minimize', 'minimize'],
                             sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(obj_multi, n_trials=10, show_progress_bar=False)
print(f'pareto front: {len(study.best_trials)} trials')
for t in study.best_trials[:3]:
    print(f'  MAE={t.values[0]:.4f}  time={t.values[1]:.2f}s  params={t.params}')

/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2

/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2

/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


pareto front: 6 trials
  MAE=21.0946  time=0.04s  params={'n_estimators': 159, 'lr': 0.14461835703563009, 'num_leaves': 123}
  MAE=21.4643  time=0.03s  params={'n_estimators': 146, 'lr': 0.10716624725460688, 'num_leaves': 74}
  MAE=51.9910  time=0.02s  params={'n_estimators': 71, 'lr': 0.010624408033089033, 'num_leaves': 109}


/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2

*When to use*: real production systems have multiple constraints (accuracy, latency, model size). *Common mistake*: combining metrics into a single weighted objective — picking weights is itself a judgement call; multi-objective makes that explicit.


---
## 6. Visualisation & post-hoc analysis

Two patterns for understanding what the search actually found.


### How do I visualise an Optuna study?

`optuna.visualization` has built-in plot functions returning Plotly figures.


In [8]:

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def obj(trial):
    x = trial.suggest_float('x', -10, 10)
    y = trial.suggest_float('y', -10, 10)
    return (x - 3) ** 2 + (y + 1) ** 2

study = optuna.create_study(sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(obj, n_trials=30, show_progress_bar=False)

# These return Plotly figures. Requires `pip install plotly` (not in base requirements).
# To render, call fig.show() or fig.write_html('out.html'). Listing them here so the
# cell runs even without plotly installed:
plot_fns = [
    'plot_optimization_history',  # objective value vs trial number
    'plot_param_importances',     # which knob mattered most
    'plot_parallel_coordinate',   # high-D structure of explored region
    'plot_slice',                 # marginal impact of each parameter
    'plot_contour',               # 2-D objective surface for any param pair
]
for name in plot_fns:
    print(name)


plot_optimization_history
plot_param_importances
plot_parallel_coordinate
plot_slice
plot_contour


*Why these plots matter*: instantly tells you whether the search converged, which parameters drove the gains, and which regions of the search space the optimiser focused on. *Common mistake*: skipping these — running 100 trials and then not looking at what they revealed.


### How do I inspect every trial's params and value?

`study.trials_dataframe()` — one row per trial with all parameters, value, state, datetime.


In [9]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def obj(trial):
    x = trial.suggest_float('x', -10, 10)
    return (x - 3) ** 2

study = optuna.create_study(sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(obj, n_trials=10, show_progress_bar=False)

df = study.trials_dataframe(attrs=('number', 'value', 'state', 'params'))
print(df)

   number      value     state  params_x
0       0   4.095483  COMPLETE  0.976270
1       1   1.699861  COMPLETE  4.303787
2       2   0.892519  COMPLETE  2.055268
3       3   4.419818  COMPLETE  0.897664
4       4  20.492860  COMPLETE -1.526904
5       5   0.006743  COMPLETE  2.917882
6       6  18.047677  COMPLETE -1.248256
7       7  23.381674  COMPLETE  7.835460
8       8  39.353731  COMPLETE  9.273255
9       9  28.421370  COMPLETE -2.331170


*Why this is useful*: post-hoc analysis (which trials succeeded, distribution of objective values, time per trial). *Common mistake*: the column for parameter `x` is `params_x` (with prefix). Filter by it: `df[df['params_x'] > 5]`.
